# 05 - Submission (v2)

Applies the pipeline tuned in notebook 04 to the test predictions of notebooks 01-03. No tuning happens here.

1. Detectors: fuse the fold models of each type (WBF, equal weights), then fuse YOLO + Faster R-CNN with the weights chosen in 04 - one pair of weights per class if 04 kept per-class weights.
2. Classifier: average `p_abn` and `p_c` over the fold models.
3. Soft combine with the tuned `alpha`, `beta`, `t_low`; partner copies only if notebook 04 kept them.
4. Map boxes back to original pixels and write `submission.csv`, with format checks.

**Inputs:** outputs of 00-04 for **all folds** + `vbd_utils` (v2). CPU is enough.

In [ ]:
# Import the shared utilities (upload vbd_utils.py as a Kaggle Dataset and attach it)
import os, sys, glob, warnings
def _locate(name, root='/kaggle/input'):
    for cur, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('png1024', 'images', 'labels', 'train', 'test')]
        if name in files:
            return cur
    raise FileNotFoundError(f'{name} not found - attach the vbd-utils dataset')
sys.path.insert(0, _locate('vbd_utils.py'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import vbd_utils as U
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
OUT = '/kaggle/working'

In [ ]:
!pip install -q ensemble-boxes
P = U.load_json(U.find_file('pipeline_params.json'))
display(P)
meta = pd.read_csv(U.find_file('meta.csv'))
PNG_DIR = U.find_dir('png1024')
test_ids = meta.loc[meta.split == 'test', 'image_id'].tolist()
sample = pd.read_csv(f'{U.competition_dir()}/sample_submission.csv')
assert set(sample.image_id) == set(test_ids)

## Test predictions of every fold model

In [ ]:
def per_fold(pattern):
    files = U.find_all(pattern)
    print(pattern, '->', len(files), 'files:', sorted(os.path.basename(f) for f in files))
    return [pd.read_csv(f) for f in files]

yolo_folds = [U.top_k_per_image(d, P['top_k']) for d in per_fold('test_yolo_fold*.csv')]
frcnn_folds = [U.top_k_per_image(d, P['top_k']) for d in per_fold('test_frcnn_fold*.csv')]
cls_folds = per_fold('test_cls_fold*.csv')
assert yolo_folds and frcnn_folds and cls_folds, 'missing test predictions'
if not (len(yolo_folds) == len(frcnn_folds) == len(cls_folds) == len(P['folds_used'])):
    print('WARNING: the number of fold models differs from the folds used in 04:', P['folds_used'])

In [ ]:
yolo_t = U.wbf_models(yolo_folds, iou_thr=P['wbf_iou'], image_ids=test_ids) if len(yolo_folds) > 1 else yolo_folds[0]
frcnn_t = U.wbf_models(frcnn_folds, iou_thr=P['wbf_iou'], image_ids=test_ids) if len(frcnn_folds) > 1 else frcnn_folds[0]
w_pc = P.get('w_per_class') or [[P['w_yolo'], P['w_frcnn']]] * 14
print('YOLO : FRCNN weights per class:', dict(zip(U.CLASSES, [f'{a:g}:{b:g}' for a, b in w_pc])))
fused_t = U.wbf_per_class([yolo_t, frcnn_t], w_pc, iou_thr=P['wbf_iou'], image_ids=test_ids)

cls_t = pd.concat(cls_folds).groupby('image_id').mean().reindex(test_ids)
assert cls_t.notna().all().all(), 'classifier predictions missing for some test images'

final_t = U.soft_combine(fused_t, cls_t, alpha=P['alpha'], beta=np.array(P['beta']), t_low=P['t_low'])
if P['use_partner']:
    final_t = U.add_partner_copies(final_t, factor=P['partner_factor'])
print('test predictions:', len(final_t))

## Sanity checks before writing
The test distribution should look like the validation distribution. Large differences point to a bug (e.g. a missing fold or a coordinate mix-up), not to a better model.

In [ ]:
oof_final = pd.read_csv(U.find_file('oof_final.csv'))
oof_cls = U.load_concat('oof_cls_fold*.csv')
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].hist(oof_cls.p_abn, bins=50, alpha=0.6, density=True, label='validation'); ax[0].hist(cls_t.p_abn, bins=50, alpha=0.6, density=True, label='test')
ax[0].set_title('p_abn'); ax[0].legend()
for a, (name, df) in zip(ax[1:], [('validation', oof_final), ('test', final_t)]):
    df[(df.class_id != 14) & (df.score >= 0.2)].class_id.map(dict(enumerate(U.CLASSES))).value_counts(normalize=True).sort_index().plot.barh(ax=a)
    a.set_title(f'{name}: class mix of boxes with score >= 0.2')
plt.tight_layout(); plt.show()
print('boxes per image (score >= 0.2): validation',
      round((oof_final[(oof_final.class_id != 14) & (oof_final.score >= 0.2)].shape[0]) / oof_final.image_id.nunique(), 2),
      '| test', round((final_t[(final_t.class_id != 14) & (final_t.score >= 0.2)].shape[0]) / len(test_ids), 2))

In [ ]:
# A few test images with their predictions (no ground truth on test)
empty_gt = pd.DataFrame(columns=['image_id', 'class_id'] + U.BOX)
top = final_t[final_t.class_id != 14].groupby('image_id').score.max().nlargest(4).index
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, i in zip(axes, top):
    U.show_predictions(i, empty_gt, final_t, PNG_DIR, score_thr=0.2, ax=ax, title=f'{i[:10]} p_abn={cls_t.p_abn[i]:.2f}')
plt.tight_layout(); plt.show()

## Write the submission

In [ ]:
strings = U.prediction_strings(final_t, meta)
sub = pd.DataFrame({'image_id': test_ids})
sub['PredictionString'] = sub.image_id.map(strings)

# Format checks
assert len(sub) == len(sample) and sub.image_id.is_unique and sub.PredictionString.notna().all()
size = meta.set_index('image_id')
for r in sub.sample(min(200, len(sub)), random_state=0).itertuples():
    v = r.PredictionString.split()
    assert len(v) % 6 == 0
    for j in range(0, len(v), 6):
        c, s, x1, y1, x2, y2 = int(v[j]), float(v[j + 1]), *map(float, v[j + 2:j + 6])
        assert 0 <= c <= 14 and 0 <= s <= 1 and x1 <= x2 and y1 <= y2
        assert x2 <= size.cols[r.image_id] + 1 and y2 <= size.rows[r.image_id] + 1
sub.to_csv(f'{OUT}/submission.csv', index=False)
print('submission.csv written | validation mAP (15 classes) of this pipeline:', round(P['oof_map15'], 4))
sub.head()